# TLE propagation with propygator

This notebook walks through Feature 1.3: fetch (or paste) a Two-Line Element set
(TLE), propagate it with SGP4/SDP4, and reuse Feature 1.1's plot + CSV output surface
wholesale.

A TLE is the compact orbit format published for tracked objects. propygator parses one
with **no JVM** (`docs/architecture.md` §10) — `fetch_tle`, `TLE.from_strings`, and the
`.epoch` / `.norad_id` accessors are all pure-Python. The Java VM spins up lazily only
at the first propagation, exactly as in `02_numerical_propagation.ipynb`.

In [ ]:
from pathlib import Path

import numpy as np

import propygator as pgr

pgr.__version__

## 1. Get a TLE

The headline entry point is `fetch_tle`, which resolves a friendly name (or a raw NORAD
catalog id) and pulls the current element set from CelesTrak, caching it on disk for
24 h so re-runs don't re-hit the network. If you're offline, fall back to
`TLE.from_strings(...)` with lines you already have — the rest of the notebook runs
either way.

In [ ]:
# A recent ISS element set, used as an offline fallback for the live fetch below.
ISS_LINE1 = "1 25544U 98067A   26171.41461525  .00008813  00000+0  16600-3 0  9990"
ISS_LINE2 = "2 25544  51.6327 284.1189 0004557 208.5194 151.5545 15.49333088572250"

try:
    tle = pgr.fetch_tle("ISS")  # also accepts a NORAD id, e.g. pgr.fetch_tle(25544)
    print("fetched live :", tle.name)
except pgr.TLEFetchError as err:
    print("offline — using a pinned ISS TLE.\n  ", err)
    tle = pgr.TLE.from_strings(ISS_LINE1, ISS_LINE2, name="ISS (ZARYA)")

Parsing is pure-Python and JVM-free — the identity and epoch come straight off the
lines. `tle.epoch` is an `Epoch` (UTC) you can compare and format like any other
propygator time.

In [ ]:
print("name     :", tle.name)
print("norad_id :", tle.norad_id)
print("epoch    :", tle.epoch.to_iso(), "UTC")
print("line1    :", tle.line1)
print("line2    :", tle.line2)

Other ways to get the same `TLE`:

- `pgr.TLE.from_strings(line1, line2, name=...)` — paste lines you already have (offline).
- `pgr.TLE.from_norad_id(25544)` — fetch by catalog number.
- `pgr.fetch_tle("HST")` / `pgr.fetch_tle("NOAA-19")` — other known names (lookup is
  case-, space-, and hyphen-insensitive), or pass any raw NORAD id.

## 2. Propagate with SGP4/SDP4

`propagate_tle` runs Orekit's SGP4/SDP4 analytic theory and returns a `Trajectory` in
**TEME** — the model's native frame, with no silent conversion (`docs/architecture.md`
§10). SGP4 (near-Earth) vs SDP4 (deep-space) is selected automatically from the mean
motion; there's no knob.

The `duration` / `output_step` sampling contract is identical to `propagate_numerical`
(both in seconds, `output_step` keyword-only), so a TLE and a numerical run with the
same arguments produce identically-gridded trajectories. `start` defaults to the TLE's
own epoch — where SGP4 is most accurate.

In [ ]:
traj = pgr.propagate_tle(tle, duration=86400, output_step=60)  # one day at 60 s
print(len(traj), "samples, in frame", traj.frame)

Like every propygator run, the `Trajectory` carries a reproducibility record in
`metadata` — here the source TLE lines (so the run is exactly reproducible), the NORAD
id, the TLE epoch, and `propagator="sgp4"` (that one token also covers the
auto-selected SDP4 branch). Note there are no force-model or integrator keys: SGP4 has
none.

In [ ]:
dict(traj.metadata)

## 3. Inspect the trajectory

The result is an ordinary `Trajectory`: it indexes and iterates as `State`s, reports
osculating Keplerian elements, and converts frames in bulk. SGP4 is native TEME, so
convert to Earth-fixed ITRF (for ground-relative work) or EME2000 with
`.to_frame(...)`.

In [ ]:
kep = traj[0].to_keplerian()
print(f"a = {kep.semi_major_axis_m / 1000:.1f} km")
print(f"e = {kep.eccentricity:.5f}")
print(f"i = {np.degrees(kep.inclination_rad):.2f} deg")

# TEME -> Earth-fixed ITRF for an Earth-relative view.
itrf = traj.to_frame(pgr.Frame.ITRF)
itrf.frame

## 4. Plot and export — Feature 1.1's surface, unchanged

Because a TLE run produces the same `Trajectory` type, every Feature 1.1 output verb
works on it unchanged. `plot_summary` stacks the ground track, altitude, and speed;
`plot_3d` returns an interactive Plotly figure.

In [ ]:
pgr.plot_summary(traj)

In [ ]:
pgr.plot_ground_track(traj)

In [ ]:
pgr.plot_3d(traj).show()  # interactive — drag to rotate, scroll to zoom

`export_csv` writes the 16 default columns plus a metadata header. Feature 1.3 added
an opt-in `mean_anomaly` group token — natural for TLE work, since mean anomaly is a
TLE field — alongside the existing additive `keplerian` and `sun` groups. `export_all`
bundles a summary PNG, an interactive 3-D HTML, and a CSV into one directory and
returns where it wrote them.

In [ ]:
pgr.export_csv(traj, Path("./iss_tle.csv"), columns=["keplerian", "mean_anomaly"])

pgr.export_all(traj, output_dir=Path("./iss_tle_run"))

## 5. A state back to a TLE (unfitted)

Going the other way, `TLE.from_state_unfitted` turns any `State` (here, a trajectory
row) into a **format-valid** TLE: correct fixed-column formatting and checksums.

It is deliberately **not round-trip-faithful** — the elements are osculating values
placed in mean-element slots, and B\* (a drag fit residual) can't be recovered from a
single state, so the rebuilt TLE won't reproduce the orbit under SGP4. The faithful
sibling is `fit_tle` (Feature 1.2, not yet built). Use this for tooling and formatting,
not accuracy.

In [ ]:
back = pgr.TLE.from_state_unfitted(traj[0], norad_id=25544)
print(back.line1)
print(back.line2)

# The lines are checksum-valid: they re-parse straight back into a TLE.
pgr.TLE.from_strings(back.line1, back.line2)

---

That's the Feature 1.3 loop: **TLE → `propagate_tle` → `Trajectory` → inspect / plot /
export**, reusing Feature 1.1's output surface end to end.

How a TLE run differs from `propagate_numerical`: SGP4/SDP4 is self-contained, so there
are **no** force-model, spacecraft, attitude, or `limits=` inputs — drag rides in the
TLE's B\* term. There's no stop-and-report either: a decay raises `TLEPropagationError`
(SGP4 is least reliable right as it approaches decay) rather than returning a partial
trajectory. And propagating far from the TLE epoch (more than ~30 days) emits a
one-time accuracy warning. The full contract — every field, the sampling grid, the
metadata grammar — is in `docs/features.md` §1.3.